# Explanation Generation
I need to use the model to get *k* explanations for each line of generated code. Firtly, I need to delete the comments line from my dataset.

In [1]:
!nvidia-smi

Mon Jun  2 20:53:26 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 555.42.06              Driver Version: 555.42.06      CUDA Version: 12.5     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A10                     Off |   00000000:17:00.0 Off |                    0 |
|  0%   30C    P8             16W /  150W |       1MiB /  23028MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
import re
import os
import glob
import torch
import pandas as pd
from tqdm import tqdm
from enum import Enum
from transformers import AutoTokenizer, AutoModelForCausalLM

In [2]:
python_csv_file = '/datasets/selected_problems_python_14B_no_comments_final.csv'
c_sharp_csv_file = '/datasets/selected_problems_c_sharp_14B_no_comments_final.csv'
typescript_csv_file = '/datasets/selected_problems_typescript_14B_no_comments_final.csv'

def get_generated_code(csv_file):
    csv_full_path = os.getcwd()+csv_file
    if os.path.exists(csv_full_path):
        return pd.read_csv(csv_full_path)
    else:
        print("File not found.")
        return None
    
python_solutions = get_generated_code(python_csv_file)
c_sharp_solutions = get_generated_code(c_sharp_csv_file)
typescript_solutions = get_generated_code(typescript_csv_file)

In [3]:
class Language(Enum):
    PYTHON = "python"
    TYPESCRIPT = "typescript"
    C_SHARP = "csharp"

In [4]:
model_name = "Qwen/Qwen2.5-Coder-14B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)#.to("cuda")

2025-06-06 16:06:01.915104: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-06-06 16:06:01.915141: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-06-06 16:06:01.915183: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-06 16:06:01.923700: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-06-06 16:06:02.992186: W tensorflow/compiler/

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

In [5]:
def add_line_numbers(code):
    lines = code.split('\n')
    numbered_lines = [f"{i + 1}: {line}" for i, line in enumerate(lines)]
    return '\n'.join(numbered_lines), len(lines)
print(add_line_numbers(python_solutions[21:22]["solution_no_comments"].to_list()[0])[0])

1: def max_beauty(n, k, w, g):
2:     gas_prefix = [0] * (n + 1)
3:     dist_prefix = [0] * (n + 1)
4:     for i in range(1, n + 1):
5:         gas_prefix[i] = gas_prefix[i - 1] + g[i - 1]
6:         if i < n:
7:             dist_prefix[i] = dist_prefix[i - 1] + w[i - 1]
8:     max_beauty = 0
9:     for l in range(1, n + 1):
10:         r = l
11:         while r <= n:
12:             total_gas_needed_forward = dist_prefix[r] - dist_prefix[l - 1] + (gas_prefix[r] - gas_prefix[l - 1])
13:             total_gas_needed_backward = dist_prefix[n] - dist_prefix[r - 1] + (gas_prefix[n] - gas_prefix[r - 1])
14:             net_gas_balance = (gas_prefix[r] - gas_prefix[l - 1]) + (gas_prefix[n] - gas_prefix[r - 1])
15:             total_gas_needed = total_gas_needed_forward + total_gas_needed_backward
16:             if net_gas_balance >= total_gas_needed:
17:                 max_beauty = max(max_beauty, r - l + 1)
18:                 break
19:             else:
20:                 additional_gas

In [6]:
def decode_outputs(explanation_outputs, explanation_prompt_length, debug = False): 
    decoded_outputs = [tokenizer.decode(output, skip_special_tokens=True)[explanation_prompt_length-1:].strip() for output in explanation_outputs]
    if debug:
        for i, explanation in enumerate(decoded_outputs):
            print(f"Explanation {i+1}:\n{explanation}\n")
    return decoded_outputs

In [7]:
max_context_length = model.config.max_position_embeddings

def get_generated_explanations(
    programming_language, problem_description, code_solution, samples=5, debug=False
):
    code_solution_with_lines, lines_number = add_line_numbers(code_solution.strip())
    explanation_prompt = f"""You will be shown a programming problem and a {programming_language.value} code solution labeled with line numbers. Your task is to explain the code **line by line**, giving exactly one explanation for **each line**.
    # Instructions:
    - The code contains exactly {lines_number} lines.
    - You must write **exactly {lines_number} explanations** — one for **each line**, from Line 1 to Line {lines_number}.
    - Every explanation must begin with this exact format: [Line x]: Explanation: ...
    - Each line starts with a number followed by ':'. Make sure for each of them you have an explanations in the end.
    - Use the format demonstrated in the example below.
    
    ## Output format:
    [Line 1]: Explanation: ...
    [Line 2]: Explanation: ...
    ...
    [Line {lines_number}]: Explanation: ...
    
    # Problem:
    {problem_description}

    # Code:
    ```python
    {code_solution_with_lines}```
    
    **Remember:** Your output must contain exactly {lines_number} explanations. If it has more or fewer, it is incorrect.
    """
    
    # if debug:
    #     print(explanation_prompt)

    explanation_inputs = tokenizer(
        explanation_prompt, return_tensors="pt", padding=True
    )
    explanation_outputs = model.generate(
        input_ids=explanation_inputs["input_ids"],
        attention_mask=explanation_inputs["attention_mask"],
        max_length=max_context_length + 1,
        # do_sample=True,
        temperature=0.7,
        # num_return_sequences=samples,
    )
    return decode_outputs(explanation_outputs, len(explanation_prompt), debug)

In [8]:
columns = ['title', 'sample_number', 'explanation']
python_explanations = pd.DataFrame([], columns=columns)
c_sharp_explanations = pd.DataFrame([], columns=columns)
typescript_explanations = pd.DataFrame([], columns=columns)

In [9]:
def add_explanations(solutions, programming_language, explanations):
    for index, row in tqdm(solutions.iterrows()):
        generated_explanations = get_generated_explanations(programming_language, row['description'], row['solution_no_comments'])
        sample_number = 1
        for explanation in generated_explanations:
            explanations.loc[len(explanations)] = [row['title'], sample_number, explanation]
            sample_number += 1

In [10]:
python_solutions[21:22]

,title,description,difficulty,source,solution,solution_no_comments
21,E. Organizing a Race,Kekoland is a country with n beautiful cities ...,Hard,CodeForces,"def max_beauty(n, k, w, g):\n # Calculate p...","def max_beauty(n, k, w, g):\n gas_prefix = ..."


In [21]:
test_explanations = get_generated_explanations(Language.PYTHON, python_solutions[21:22]['description'].tolist()[0], python_solutions[21:22]['solution_no_comments'].tolist()[0], 2, True)

Explanation 1:
[Line 1]: Explanation: Define a function `max_beauty` that takes four parameters: `n` (number of cities), `k` (number of presents), `w` (list of road lengths), and `g` (list of gas amounts).
    [Line 2]: Explanation: Initialize a list `gas_prefix` of size `n + 1` with all elements set to 0. This list will store the prefix sums of gas amounts.
    [Line 3]: Explanation: Initialize a list `dist_prefix` of size `n + 1` with all elements set to 0. This list will store the prefix sums of road distances.
    [Line 4]: Explanation: Start a loop that iterates from 1 to `n` (inclusive) to fill the prefix sum arrays.
    [Line 5]: Explanation: Update the `gas_prefix` array at index `i` by adding the gas amount at index `i - 1` to the previous value.
    [Line 6]: Explanation: Check if the current index `i` is less than `n` to avoid index out of range error.
    [Line 7]: Explanation: Update the `dist_prefix` array at index `i` by adding the road distance at index `i - 1` to the p

In [33]:
add_explanations(python_solutions[0:1], Language.PYTHON, python_explanations)

1it [42:05, 2525.58s/it]


In [34]:
python_explanations

,title,sample_number,explanation
0,Sports Rank,1,``` ``` [Line 1]: Explanation: Define a functi...
1,Sports Rank,2,``` [Line 1]: Explanation: Define a function n...


In [35]:
print(python_explanations[0:1]["explanation"].to_list()[0])

``` ``` [Line 1]: Explanation: Define a function named `read_input` that does not take any parameters.
[Line 2]: Explanation: Import the `sys` module to access system-specific parameters and functions.
[Line 3]: Explanation: Assign the `read` method of `sys.stdin` to the variable `input`. This allows reading from standard input.
[Line 4]: Explanation: Read all input data as a single string and split it into a list of strings `data` using whitespace as the delimiter.
[Line 5]: Explanation: Initialize a variable `index` to 0. This will be used to keep track of the current position in the `data` list.
[Line 6]: Explanation: Convert the first element of `data` (at index 0) to an integer and assign it to the variable `d`. This represents the number of data sets.
[Line 7]: Explanation: Increment `index` by 1 to move to the next element in the `data` list.
[Line 8]: Explanation: Initialize an empty list `datasets` to store the scores for each data set.
[Line 9]: Explanation: Start a loop that

**Now, I need to run these methods for each generated solution.**

In [43]:
columns = ['title', 'sample_number', 'explanation']
python_explanations = pd.DataFrame([], columns=columns)
c_sharp_explanations = pd.DataFrame([], columns=columns)
typescript_explanations = pd.DataFrame([], columns=columns)

In [58]:
python_solutions[21:]

,title,description,difficulty,source,solution,solution_no_comments
21,E. Organizing a Race,Kekoland is a country with n beautiful cities ...,Hard,CodeForces,"def max_beauty(n, k, w, g):\n # Calculate p...","def max_beauty(n, k, w, g):\n gas_prefix = ..."
22,A+B Again?,"Given a two-digit positive integer n, find the...",Easy,CodeForces,# Read the number of test cases\nt = int(input...,t = int(input())\nfor _ in range(t):\n n = ...
23,Party,People working in MDCS (Microsoft Development ...,Medium,CodeForces,"def max_happiness(N, friday_matrix, saturday_m...","def max_happiness(N, friday_matrix, saturday_m..."
24,Zebra-like Numbers,We call a positive integer zebra-like if its b...,Medium,CodeForces,def is_zebra_like(n):\n # Check if n is zeb...,def is_zebra_like(n):\n bin_rep = []\n w...
25,Morse Code,Morse code is a classical way to communicate o...,Hard,CodeForces,"class Node:\n def __init__(self, char=None,...","class Node:\n def __init__(self, char=None,..."
26,Set of Strangers,You are given a table of n rows and m columns....,Easy,CodeForces,"def min_steps_to_uniform_color(t, test_cases):...","def min_steps_to_uniform_color(t, test_cases):..."
27,Amusing numbers,Amusing numbers are numbers consisting only of...,Medium,SPOJ,def kth_amusing_number(k):\n # Convert k-1 ...,def kth_amusing_number(k):\n binary_represe...
28,Time to live,"As you might know, most computer networks are ...",Easy,SPOJ,"def dfs(node, parent, graph, depths):\n max...","def dfs(node, parent, graph, depths):\n max..."
29,New Distinct Substrings,"Given a string, we need to find the total numb...",Hard,SPOJ,class TrieNode:\n def __init__(self):\n ...,class TrieNode:\n def __init__(self):\n ...


In [59]:
add_explanations(python_solutions[21:], Language.PYTHON, python_explanations)

9it [12:22:26, 4949.62s/it]


In [60]:
python_explanations

,title,sample_number,explanation
0,Marathon,1,``` [Line 1]: Explanation: The entry point of ...
1,Marathon,2,``` [Line 1]: Explanation: The main function i...
2,Marathon,3,[Line 1]: Explanation: Define the main functio...
3,Marathon,4,[Line 1]: Explanation: The main function is de...
4,Marathon,5,[Line 1]: Explanation: Defines the main functi...
...,...,...,...
115,New Distinct Substrings,1,``` [Line 1]: Explanation: Define a class name...
116,New Distinct Substrings,2,``` [Line 1]: Explanation: Define a class name...
117,New Distinct Substrings,3,[Line 1]: Explanation: Defines a class named T...
118,New Distinct Substrings,4,[Line 1]: Explanation: Define a class named `T...


In [61]:
python_explanations.to_csv('datasets/python_14B_explanations_final_sample_5_4.csv', index=False)

In [91]:
add_explanations(c_sharp_solutions, Language.C_SHARP, c_sharp_explanations)

30it [6:59:36, 839.20s/it] 


In [92]:
add_explanations(typescript_solutions, Language.TYPESCRIPT, typescript_explanations)

30it [6:32:44, 785.47s/it]


# Merge Explanation Files
I should merge the files `datasets/python_14B_explanations_final_sample_5_*`.

In [8]:
file_paths = glob.glob('datasets/python_14B_explanations_final_sample_5_*.csv')
file_paths

['datasets/python_14B_explanations_final_sample_5_1.csv',
 'datasets/python_14B_explanations_final_sample_5_2.csv',
 'datasets/python_14B_explanations_final_sample_5_3.csv',
 'datasets/python_14B_explanations_final_sample_5_4.csv']

In [9]:
df_list = [pd.read_csv(file) for file in file_paths]
combined_df = pd.concat(df_list, ignore_index=True)
python_explanations_final = combined_df.drop_duplicates()
python_explanations_final.shape

(150, 3)

In [10]:
python_explanations_final.to_csv('datasets/python_14B_explanations_5_samples.csv', index=False)

# Line Explanation Number Checking
To make sure that each line was explained, I need to check the number of code lines with the number of explanation lines.

In [11]:
python_explanations_final = pd.read_csv('datasets/python_14B_explanations_5_samples_refined_1.csv')

In [12]:
def check_explanations_for_lines(code, explanations, debug = False):
    code_lines = [line for line in code.split('\n') if line.strip()]
    explanation_lines = [line for line in explanations.split('\n') if "Line" in line.strip()]
    if debug:
        print(len(code_lines))
        print(len(explanation_lines))
    return len(code_lines) == len(explanation_lines)

In [13]:
check_explanations_for_lines(
    python_solutions[0:1]["solution_no_comments"].to_list()[0],
    python_explanations_final[0:1]["explanation"].to_list()[0],
)

True

In [14]:
def check_all_explanations(solutions, explanations):
    wrong_explanations = {}
    for index, row in tqdm(solutions.iterrows()):
        for sample in range(5):
            if check_explanations_for_lines(
                row["solution_no_comments"],
                explanations[explanations["title"] == row["title"]]
                .reset_index()
                .loc[sample, "explanation"],
            ):
                continue
            else:
                if row['title'] in wrong_explanations:
                    wrong_explanations[row['title']].append(sample + 1)
                else:
                    wrong_explanations[row['title']] = [sample + 1]
                # print(f"{sample}: {row['title']}") # \n{explanations[explanations['title'] == row['title']].reset_index().loc[sample, 'explanation']}")
    return wrong_explanations

In [32]:
wrong_explanations = check_all_explanations(python_solutions, python_explanations_final)
wrong_explanations

30it [00:00, 168.09it/s]


{}

In [28]:
while len(wrong_explanations) != 0:
    for wrong_explanation_key, wrong_explanation_values in wrong_explanations.items():
        solution = python_solutions[
            python_solutions["title"] == wrong_explanation_key
        ].iloc[0]
        new_explanations = get_generated_explanations(
            Language.PYTHON,
            solution["description"],
            solution["solution_no_comments"],
            len(wrong_explanation_values),
        )

        for index, new_explanation in enumerate(new_explanations):
            python_explanations_final.loc[
                (python_explanations_final["title"] == wrong_explanation_key)
                & (
                    python_explanations_final["sample_number"]
                    == wrong_explanation_values[index]
                ),
                "explanation",
            ] = new_explanation

    wrong_explanations = check_all_explanations(python_solutions, python_explanations_final)

30it [00:00, 351.66it/s]
30it [00:00, 290.58it/s]
30it [00:00, 292.62it/s]
30it [00:00, 334.51it/s]


In [29]:
wrong_explanations = check_all_explanations(python_solutions, python_explanations_final)
wrong_explanations

30it [00:00, 356.77it/s]


{}

In [31]:
python_explanations_final.to_csv('datasets/python_14B_explanations_5_samples_refined_1.csv', index=False)

In [93]:
python_explanations.to_csv('datasets/python_14B_explanations_final_1.csv', index=False)

In [94]:
c_sharp_explanations.to_csv('datasets/c_sharp_14B_explanations_final_1.csv', index=False)

In [95]:
typescript_explanations.to_csv('datasets/typescript_14B_explanations_final_1.csv', index=False)

In [32]:
# while len(wrong_explanations) != 0:
for wrong_explanation_key, wrong_explanation_values in wrong_explanations.items():
    solution = python_solutions[
        python_solutions["title"] == wrong_explanation_key
    ].iloc[0]

    for value in wrong_explanation_values:
        new_explanation = get_generated_explanations(
            Language.PYTHON,
            solution["description"],
            solution["solution_no_comments"],
            len(wrong_explanation_values),
        )

        python_explanations_final.loc[
            (python_explanations_final["title"] == wrong_explanation_key)
            & (python_explanations_final["sample_number"] == value),
            "explanation",
        ] = new_explanation

    # wrong_explanations = check_all_explanations(python_solutions[:15], python_explanations_final)